# 🐦‍⬛ Karga-2B-Thinking — Çalışan Notebook
Türkçe düşünen, adım adım çözen küçük model.

In [ ]:
# 1. Gereksinimleri Yükle
!pip install -q transformers accelerate torch

In [ ]:
# 2. Model ve Tokenizer'ı Yükle
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "ilkayO/Karga-2B-Thinking"

print("Tokenizer yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Model yükleniyor (bu biraz sürebilir)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

device = next(model.parameters()).device
print(f"✅ Model hazır! Cihaz: {device}")

In [ ]:
# 3. Yardımcı Fonksiyon: Yanıt Üret

def karga_sor(soru: str, max_new_tokens: int = 1024) -> str:
    messages = [
        {"role": "system", "content": "Adın Karga. Soruları mantıklı ve adım adım düşünerek yanıtla."},
        {"role": "user", "content": soru}
    ]

    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    terminators = [tokenizer.eos_token_id]
    if "<|eot_id|>" in tokenizer.get_vocab():
        terminators.append(tokenizer.convert_tokens_to_ids("<|eot_id|>"))

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=0.6,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            eos_token_id=terminators,
            pad_token_id=tokenizer.eos_token_id
        )

    new_ids = output_ids[0][inputs.input_ids.shape[-1]:]
    full_response = tokenizer.decode(new_ids, skip_special_tokens=True)
    return full_response


def yaniti_goster(soru: str):
    """
    Soruyu sor, düşünce ve cevabı güzel formatta yazdır.
    """
    print(f"\n❓ Soru: {soru}")
    print("-" * 60)

    yanit = karga_sor(soru)

    if "</think>" in yanit:
        parcalar = yanit.split("</think>")
        dusunce = parcalar[0].replace("<think>", "").strip()
        cevap = parcalar[1].strip() if len(parcalar) > 1 else ""

        print("\n💭 Düşünce Süreci:")
        print(dusunce)
        print("\n✅ Cevap:")
        print(cevap)
    else:
        print("\n✅ Yanıt:")
        print(yanit)

    return yanit

print("✅ Fonksiyonlar hazır!")

In [ ]:
# 4. Test 1: Matematik
yaniti_goster("Bir tarlada 5 koyun ve 3 tavuk vardır. Toplam kaç ayak vardır? Adım adım hesapla.")

In [ ]:
# 5. Test 2: Mantık
yaniti_goster("Eğer bugün salı ise, 15 gün sonra hangi gün olur?")

In [ ]:
# 6. Test 3: Kodlama
yaniti_goster("İki sayıyı toplayan 'sum_two' adında bir Python fonksiyonu yazın.")

In [ ]:
# 7. Kendi Sorunuzu Sorun
sorum = "Türkiye'nin başkenti nedir ve nüfusu kaçtır?"  # <-- buraya yaz
yaniti_goster(sorum)

In [ ]:
# 8. (Opsiyonel) Streaming ile Gerçek Zamanlı Yanıt
from transformers import TextIteratorStreamer
from threading import Thread

def karga_streaming(soru: str):
    messages = [
        {"role": "system", "content": "Adın Karga. Soruları mantıklı ve adım adım düşünerek yanıtla."},
        {"role": "user", "content": soru}
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    generation_kwargs = {
        "input_ids": inputs.input_ids,
        "attention_mask": inputs.attention_mask,
        "streamer": streamer,
        "max_new_tokens": 1024,
        "temperature": 0.6,
        "top_p": 0.9,
        "repetition_penalty": 1.1,
        "do_sample": True
    }

    print(f"\n❓ Soru: {soru}")
    print("-" * 60)

    Thread(target=model.generate, kwargs=generation_kwargs).start()

    tam_yanit = ""
    for token in streamer:
        print(token, end="", flush=True)
        tam_yanit += token

    print()
    return tam_yanit

# Streaming testi
karga_streaming("Python'da bir liste nasıl sıralanır? Örneklerle anlat.")